In [1]:
# Install temporalio Python SDK
!pip install temporalio nest_asyncio

# Download and install the official Temporal CLI for Linux
!curl -sSf https://temporal.download/cli.sh | sh

# Move binary to /usr/local/bin so it is available globally
!sudo cp /root/.temporalio/bin/temporal /usr/local/bin/
!temporal --version

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.6/107.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.4/86.4 kB 8.5 MB/s eta 0:00:00
temporal: Downloading Temporal CLI latest
temporal: Temporal CLI installed at /root/.temporalio/bin/temporal
temporal: For convenience, we recommend adding it to your PATH
temporal: If using bash, run echo export PATH="\$PATH:/root/.temporalio/bin" >> ~/.bashrc
temporal version 1.8.3 (Server 1.31.2, UI 2.50.1)


In [2]:
import subprocess
import time

# Start temporal server in the background (runs on localhost:7233)
server_proc = subprocess.Popen(
    ["temporal", "server", "start-dev", "--headless"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait a few seconds for the gRPC server to spin up
time.sleep(5)
print("Temporal server process ID:", server_proc.pid)

Temporal server process ID: 3836


In [4]:
import asyncio
import nest_asyncio
from datetime import timedelta
from temporalio import activity, workflow
from temporalio.client import Client
from temporalio.worker import Worker, UnsandboxedWorkflowRunner

# Enable nested event loops inside Jupyter/Colab
nest_asyncio.apply()

# 1. Define an Activity
@activity.defn
async def say_hello_activity(name: str) -> str:
    return f"Hello, {name}! Durable execution works in Colab."

# 2. Define a Workflow
@workflow.defn
class GreetingWorkflow:
    @workflow.run
    async def run(self, name: str) -> str:
        return await workflow.execute_activity(
            say_hello_activity,
            name,
            start_to_close_timeout=timedelta(seconds=10),
        )

async def main():
    # 3. Connect to the local Temporal cluster
    client = await Client.connect("localhost:7233")

    # 4. Start a Worker with the Sandbox disabled for Python 3.13 compatibility
    task_queue = "colab-task-queue"
    worker = Worker(
        client,
        task_queue=task_queue,
        workflows=[GreetingWorkflow],
        activities=[say_hello_activity],
        workflow_runner=UnsandboxedWorkflowRunner(),
    )

    # Run worker in the background
    worker_task = asyncio.create_task(worker.run())

    try:
        # 5. Execute the workflow
        result = await client.execute_workflow(
            GreetingWorkflow.run,
            "Colab User",
            id="colab-first-workflow",
            task_queue=task_queue,
        )
        print("\n--- Workflow Execution Result ---")
        print("Output:", result)
        print("---------------------------------")
    finally:
        # Stop worker once complete
        worker_task.cancel()
        try:
            await worker_task
        except asyncio.CancelledError:
            pass

# Execute the runner
await main()


--- Workflow Execution Result ---
Output: Hello, Colab User! Durable execution works in Colab.
---------------------------------


In [ ]:
import sys
print(sys.version)

3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]


In [5]:
import asyncio
import base64
import json
import os
from datetime import timedelta
from typing import List

import nest_asyncio
import requests
from google.colab import userdata
from temporalio import activity, workflow
from temporalio.client import Client
from temporalio.common import RetryPolicy
from temporalio.worker import UnsandboxedWorkflowRunner, Worker

nest_asyncio.apply()

# Configure environment credentials
os.environ["EBAY_CLIENT_ID"] = userdata.get("EBAY_CLIENT_ID") or ""
os.environ["EBAY_CLIENT_SECRET"] = userdata.get("EBAY_CLIENT_SECRET") or ""
os.environ["EBAY_BASE_URL"] = userdata.get("EBAY_BASE_URL") or "https://api.ebay.com"

# Common retry policy for network & API activities
DEFAULT_RETRY_POLICY = RetryPolicy(
    initial_interval=timedelta(seconds=2),
    backoff_coefficient=2.0,
    maximum_interval=timedelta(seconds=30),
    maximum_attempts=4,
)

# ---------------------------------------------------------------------------
# Activities
# ---------------------------------------------------------------------------


@activity.defn
async def fetch_oauth_token_activity() -> str:
    """Acquires eBay OAuth 2.0 application access token."""
    client_id = os.environ["EBAY_CLIENT_ID"]
    client_secret = os.environ["EBAY_CLIENT_SECRET"]
    base_url = os.environ["EBAY_BASE_URL"]

    auth_url = f"{base_url}/identity/v1/oauth2/token"
    credentials = f"{client_id}:{client_secret}"
    b64_credentials = base64.b64encode(credentials.encode("utf-8")).decode("utf-8")

    headers = {
        "Content-Type": "application/x-www-form-urlencoded",
        "Authorization": f"Basic {b64_credentials}",
    }
    data = {
        "grant_type": "client_credentials",
        "scope": "https://api.ebay.com/oauth/api_scope",
    }

    response = requests.post(auth_url, headers=headers, data=data, timeout=15)
    response.raise_for_status()
    return response.json()["access_token"]


@activity.defn
async def read_lines_from_file_activity(file_path: str) -> List[str]:
    """Reads queries or IDs from a text file, ignoring blank lines."""
    if not os.path.exists(file_path):
        return []
    with open(file_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


@activity.defn
async def search_ebay_activity(params: dict) -> List[str]:
    """Executes Browse API search and extracts solely item IDs."""
    token = params["token"]
    query = params["query"]
    limit = params.get("limit", 3)
    base_url = os.environ["EBAY_BASE_URL"]

    endpoint = f"{base_url}/buy/browse/v1/item_summary/search"
    headers = {
        "Authorization": f"Bearer {token}",
        "X-EBAY-C-MARKETPLACE-ID": "EBAY_US",
        "Accept": "application/json",
    }

    response = requests.get(
        endpoint, headers=headers, params={"q": query, "limit": limit}, timeout=15
    )
    response.raise_for_status()

    payload = response.json()
    items = payload.get("itemSummaries", [])
    return [item["itemId"] for item in items if "itemId" in item]


@activity.defn
async def append_items_to_file_activity(params: dict) -> int:
    """Appends discovered item IDs to an external text sink."""
    file_path = params["file_path"]
    item_ids = params["item_ids"]

    # Avoid duplicate lines
    existing_ids = set()
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            existing_ids = {line.strip() for line in f if line.strip()}

    new_ids = [i for i in item_ids if i not in existing_ids]
    if new_ids:
        with open(file_path, "a", encoding="utf-8") as f:
            for item_id in new_ids:
                f.write(f"{item_id}\n")

    return len(new_ids)


@activity.defn
async def fetch_item_details_activity(params: dict) -> dict:
    """Retrieves full metadata for an individual item ID."""
    token = params["token"]
    item_id = params["item_id"]
    base_url = os.environ["EBAY_BASE_URL"]

    endpoint = f"{base_url}/buy/browse/v1/item/{item_id}"
    headers = {
        "Authorization": f"Bearer {token}",
        "X-EBAY-C-MARKETPLACE-ID": "EBAY_US",
        "Accept": "application/json",
    }

    response = requests.get(endpoint, headers=headers, timeout=15)
    response.raise_for_status()
    return response.json()


@activity.defn
async def save_item_json_activity(params: dict) -> str:
    """Saves raw product payload into a target directory."""
    item_id = params["item_id"]
    data = params["data"]
    output_dir = params.get("output_dir", "./ebay_items")

    os.makedirs(output_dir, exist_ok=True)
    clean_id = item_id.replace("|", "_")
    target_path = os.path.join(output_dir, f"{clean_id}.json")

    with open(target_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

    return target_path


# ---------------------------------------------------------------------------
# Workflow 1: Search Pipeline
# ---------------------------------------------------------------------------


@workflow.defn
class EbaySearchPipelineWorkflow:
    @workflow.run
    async def run(self, input_queries_file: str, output_items_file: str) -> dict:
        # Step 1: Read queries from input text file
        queries = await workflow.execute_activity(
            read_lines_from_file_activity,
            input_queries_file,
            start_to_close_timeout=timedelta(seconds=10),
        )
        if not queries:
            return {"status": "skipped", "reason": "No queries found in file"}

        # Step 2: Acquire Auth Token
        token = await workflow.execute_activity(
            fetch_oauth_token_activity,
            start_to_close_timeout=timedelta(seconds=15),
            retry_policy=DEFAULT_RETRY_POLICY,
        )

        # Step 3: Run searches
        collected_ids = []
        for q in queries:
            ids = await workflow.execute_activity(
                search_ebay_activity,
                {"token": token, "query": q, "limit": 3},
                start_to_close_timeout=timedelta(seconds=20),
                retry_policy=DEFAULT_RETRY_POLICY,
            )
            collected_ids.extend(ids)

        # Step 4: Write discovered IDs to the intermediate output file
        written_count = await workflow.execute_activity(
            append_items_to_file_activity,
            {"file_path": output_items_file, "item_ids": collected_ids},
            start_to_close_timeout=timedelta(seconds=10),
        )

        return {
            "status": "completed",
            "queries_processed": len(queries),
            "new_items_saved": written_count,
            "target_file": output_items_file,
        }


# ---------------------------------------------------------------------------
# Workflow 2: Item Enrichment Pipeline
# ---------------------------------------------------------------------------


@workflow.defn
class EbayItemEnrichmentWorkflow:
    @workflow.run
    async def run(self, input_items_file: str, output_dir: str = "./ebay_items") -> dict:
        # Step 1: Read item IDs from file (can come from Workflow 1 or manual text file)
        item_ids = await workflow.execute_activity(
            read_lines_from_file_activity,
            input_items_file,
            start_to_close_timeout=timedelta(seconds=10),
        )
        if not item_ids:
            return {"status": "skipped", "reason": "No item IDs to process"}

        # Step 2: Acquire Auth Token
        token = await workflow.execute_activity(
            fetch_oauth_token_activity,
            start_to_close_timeout=timedelta(seconds=15),
            retry_policy=DEFAULT_RETRY_POLICY,
        )

        # Step 3: Fetch each item's details and save them
        saved_paths = []
        for item_id in item_ids:
            item_data = await workflow.execute_activity(
                fetch_item_details_activity,
                {"token": token, "item_id": item_id},
                start_to_close_timeout=timedelta(seconds=20),
                retry_policy=DEFAULT_RETRY_POLICY,
            )

            path = await workflow.execute_activity(
                save_item_json_activity,
                {"item_id": item_id, "data": item_data, "output_dir": output_dir},
                start_to_close_timeout=timedelta(seconds=10),
            )
            saved_paths.append(path)

        return {
            "status": "completed",
            "items_processed": len(saved_paths),
            "output_dir": output_dir,
        }


# ---------------------------------------------------------------------------
# Runner Execution
# ---------------------------------------------------------------------------


async def main():
    client = await Client.connect("localhost:7233")
    task_queue = "ebay-processing-queue"

    # Setup test queries file
    queries_path = "search_queries.txt"
    items_path = "item_ids.txt"

    with open(queries_path, "w", encoding="utf-8") as f:
        f.write("mechanical keyboard\ngaming mouse\n")

    worker = Worker(
        client,
        task_queue=task_queue,
        workflows=[EbaySearchPipelineWorkflow, EbayItemEnrichmentWorkflow],
        activities=[
            fetch_oauth_token_activity,
            read_lines_from_file_activity,
            search_ebay_activity,
            append_items_to_file_activity,
            fetch_item_details_activity,
            save_item_json_activity,
        ],
        workflow_runner=UnsandboxedWorkflowRunner(),
    )

    worker_task = asyncio.create_task(worker.run())

    try:
        # 1. Run Workflow 1: Search Pipeline
        print("\n--- Executing Workflow 1 (Search) ---")
        search_result = await client.execute_workflow(
            EbaySearchPipelineWorkflow.run,
            args=[queries_path, items_path],
            id="ebay-search-run-001",
            task_queue=task_queue,
        )
        print("Workflow 1 Output:", search_result)

        # 2. Run Workflow 2: Enrichment Pipeline (reads from the file written by Workflow 1)
        print("\n--- Executing Workflow 2 (Enrichment) ---")
        enrich_result = await client.execute_workflow(
            EbayItemEnrichmentWorkflow.run,
            args=[items_path, "./ebay_item_jsons"],
            id="ebay-enrich-run-001",
            task_queue=task_queue,
        )
        print("Workflow 2 Output:", enrich_result)

    finally:
        worker_task.cancel()
        try:
            await worker_task
        except asyncio.CancelledError:
            pass


# Execute in Colab
await main()


--- Executing Workflow 1 (Search) ---
Workflow 1 Output: {'new_items_saved': 6, 'queries_processed': 2, 'status': 'completed', 'target_file': 'item_ids.txt'}

--- Executing Workflow 2 (Enrichment) ---
Workflow 2 Output: {'items_processed': 6, 'output_dir': './ebay_item_jsons', 'status': 'completed'}
